### Lab2 - Implementation of a Language Model (RNN and Transformer-based)

In this notebook we design, implement and evaluate two small causal language models: an LSTM and a decoder-only Transformer (both with about 10M parameters) on Polish text and compare them in terms of perplexity and time efficiency (training and inference). 

The experiment was run on a laptop with an Intel Core i7-13700H CPU (14 cores, 20 threads), 30 GB of RAM and an NVIDIA GeForce RTX 4070 Laptop GPU with 8 GB of VRAM, under Ubuntu Linux (Python 3.14, PyTorch 2.15 with CUDA).

### Data preparation
For this experiment, we use `plwiki` dataset from Speakleash.

The vocabulary size ($16384 \approx 2^{14}$, we want to use the GPU in the most optimal way) and the sequence length (256) are fixed here, because they determine the size of the models. The batch size is chosen later, after the models are built, by checking how much GPU memory they use.

In [1]:
import json
import random
import re
import os, platform
from pathlib import Path

import numpy as np
import sentencepiece as spm
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

/home/janwo/Dokumenty/computational-linguistics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
VOCAB_SIZE = 16384
SEQ_LEN = 256

# data
SOURCE = "plwiki"
KEEP_PROB = 0.15         # we keep 15% of the documents randomly
MIN_CHARS = 200          # skip very short documents
TEST_FRAC = 0.01        # fraction of the documents for the held-out test set
VAL_FRAC = 0.005         # fraction of the documents for the validation set (used during training)

SPEAKLEASH_DIR = Path("speakleash_data")
DATA_DIR = Path("datasets")
DATA_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

torch: 2.15.0+cu132
device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM: 8.2 GB


Training data comes from the Speakleash Polish corpus. We use the Polish Wikipedia (`plwiki`) part and split it by documents into train / validation / test.

The beginning of the file is ordered (the first articles start with the letter A), so we iterate through the whole file and keep each article with probability `KEEP_PROB`. Then the whitespace is normalized, duplicates and very short documents are removed and the documents are shuffled.

The first documents after shuffling form the test set (held-out Wikipedia articles, used for the final perplexity), the next ones the validation set (used to monitor training), and the rest is the training data. The splits are made by documents, as fractions of all kept documents (`TEST_FRAC`, `VAL_FRAC`). The result is saved to `.jsonl` files, so the reading of the dump is done only once.

In [3]:
splits = ["train", "val", "test"]
paths = {s: DATA_DIR / f"{s}.jsonl" for s in splits}

if all(p.exists() for p in paths.values()):
    print("split files already exist, skipping")
else:
    from speakleash import Speakleash

    sl = Speakleash(str(SPEAKLEASH_DIR))
    rng = random.Random(SEED)

    docs = []
    for doc in tqdm(sl.get(SOURCE).data, desc="reading documents"):
        if rng.random() < KEEP_PROB:
            docs.append(" ".join(doc.split()))      # normalization, only spaces

    docs = [d for d in dict.fromkeys(docs) if len(d) >= MIN_CHARS]   # no duplicates and very short ones
    rng.shuffle(docs)

    n_test = int(len(docs) * TEST_FRAC)
    n_val = int(len(docs) * VAL_FRAC)
    test_docs = docs[:n_test]
    val_docs = docs[n_test:n_test + n_val]
    train_docs = docs[n_test + n_val:]

    for s, split_docs in zip(splits, [train_docs, val_docs, test_docs]):
        with open(paths[s], "w", encoding="utf-8") as f:
            for d in split_docs:
                f.write(json.dumps(d, ensure_ascii=False) + "\n")

split files already exist, skipping


In [4]:
def read_docs(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

print(f"{'split':<6} {'docs':>8} {'chars':>13} {'words':>12}")
for s in splits:
    n_docs = n_chars = n_words = 0
    for text in read_docs(paths[s]):
        n_docs += 1
        n_chars += len(text)
        n_words += len(text.split())
    print(f"{s:<6} {n_docs:>8} {n_chars:>13,} {n_words:>12,}")

first = next(read_docs(paths["train"]))
print()
print(first[:500])

split      docs         chars        words
train    174212   273,560,204   37,837,752
val         879     1,286,290      178,421
test        879     1,252,674      173,323

Rosetta, Rosette ("arab." Raszid) – miasto portowe w Egipcie nad jednym z ramion delty Nilu o tej samej nazwie, w pobliżu Morza Śródziemnego, 65 km na wschód od Aleksandrii, w muhafazie Al-Buhajra. Założone ok. 870 roku. W 1799 w Rosetcie znaleziono granitową płytę z trójjęzycznym napisem tzw. kamień z Rosetty, który stał się podstawą do odczytania hieroglifów. W mieście rozwinął się przemysł spożywczy, włókienniczy oraz tytoniowy.


### Tokenizer

We use SentencePiece with the BPE algorithm and a vocabulary of 16384 tokens:
- subword units avoid the huge vocabulary (and many unknown words) that a word-level tokenizer would have for a highly inflected language like Polish,
- 16k is a compromise: a smaller vocabulary gives longer sequences, a larger one takes too many parameters from a small model (the embedding matrix has `VOCAB_SIZE * d_model` parameters),
- `byte_fallback=True` means there are no unknown tokens: rare characters are split into bytes.

The tokenizer is trained only on the training split. An end-of-text token (`</s>`) is added after every document.

In [5]:
MODEL_PREFIX = str(DATA_DIR / "spm_bpe_16k")

def sentences(path):
    # SentencePiece expects short lines, so the documents are split into sentences
    for text in read_docs(path):
        yield from re.split(r"(?<=[.!?]) ", text)

if not Path(MODEL_PREFIX + ".model").exists():
    spm.SentencePieceTrainer.train(
        sentence_iterator=sentences(paths["train"]),
        model_prefix=MODEL_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type="bpe",
        character_coverage=1.0,
        byte_fallback=True,
        max_sentence_length=16384,
        unk_id=0, bos_id=-1, eos_id=1, pad_id=-1,
        num_threads=16,
    )

sp = spm.SentencePieceProcessor(model_file=MODEL_PREFIX + ".model")
assert sp.get_piece_size() == VOCAB_SIZE
EOS_ID = sp.eos_id()
print("vocab size:", sp.get_piece_size(), "| eos id:", EOS_ID)

example = "AGH jest największą uczelnią techniczną w Polsce."
pieces = sp.encode(example, out_type=str)
print(pieces)
print(sp.decode(sp.encode(example)))

vocab size: 16384 | eos id: 1
['▁A', 'G', 'H', '▁jest', '▁największą', '▁uczelni', 'ą', '▁techni', 'czną', '▁w', '▁Polsce', '.']
AGH jest największą uczelnią techniczną w Polsce.


The documents vary in length, whereas the network expects fixed-shape batches. Instead of padding short documents, we concatenate all of them into a single continuous sequence. Every split is converted into one long stream of token ids (documents separated by `</s>`) and saved as a `.npy` file. For the test split we also keep the number of words and characters, because they are needed later for perplexity per word, which does not depend on the tokenizer.

In [6]:
def encode_split(split, chunk_docs=1000):
    ids, n_words, n_chars, batch = [], 0, 0, []

    def flush():
        for toks in sp.encode(batch):
            ids.append(np.array(toks + [EOS_ID], dtype=np.uint16))

    for text in tqdm(read_docs(paths[split]), desc=f"encoding {split}"):
        batch.append(text)
        n_words += len(text.split())
        n_chars += len(text)
        if len(batch) == chunk_docs:
            flush()
            batch = []
    if batch:
        flush()

    ids = np.concatenate(ids)
    return ids, {"tokens": len(ids), "words": n_words, "chars": n_chars}

tokens, meta = {}, {}
for s in splits:
    npy, meta_file = DATA_DIR / f"{s}_ids.npy", DATA_DIR / f"{s}_meta.json"
    if npy.exists() and meta_file.exists():
        tokens[s] = np.load(npy)
        meta[s] = json.loads(meta_file.read_text())
    else:
        tokens[s], meta[s] = encode_split(s)
        np.save(npy, tokens[s])
        meta_file.write_text(json.dumps(meta[s]))

print(f"{'split':<6} {'tokens':>13} {'chars/token':>12} {'tokens/word':>12}")
for s in splits:
    m = meta[s]
    print(f"{s:<6} {m['tokens']:>13,} {m['chars'] / m['tokens']:>12.2f} {m['tokens'] / m['words']:>12.2f}")

split         tokens  chars/token  tokens/word
train     70,926,139         3.86         1.87
val          337,659         3.81         1.89
test         326,975         3.83         1.89


Training examples are non-overlapping windows of `SEQ_LEN + 1` tokens: the input is the first `SEQ_LEN` tokens and the target is the same window shifted by one token (the next token at every position).

In [7]:
class LMDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = ids
        self.seq_len = seq_len
        self.n = (len(ids) - 1) // seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        start = i * self.seq_len
        chunk = torch.from_numpy(self.ids[start:start + self.seq_len + 1].astype(np.int64))
        return chunk[:-1], chunk[1:]

train_ds = LMDataset(tokens["train"], SEQ_LEN)
val_ds = LMDataset(tokens["val"], SEQ_LEN)
test_ds = LMDataset(tokens["test"], SEQ_LEN)
print("sequences (train / val / test):", len(train_ds), len(val_ds), len(test_ds))

x, y = train_ds[0]
print(x.shape, y.shape)
print(sp.decode(x[:30].tolist()))

sequences (train / val / test): 277055 1318 1277
torch.Size([256]) torch.Size([256])
Rosetta, Rosette ("arab." Raszid) – miasto portowe w Egipcie nad jednym z ramion delty Ni


The data is ready. The `DataLoader`s will be created in the training section, once the batch size is chosen.

### Models

Both models are causal language models with about 10M parameters and the same input/output embedding matrix (weight tying), so the comparison is fair:

- **Transformer**: decoder-only, pre-LayerNorm, learned positional embeddings, causal self-attention, 8 layers, `d_model=256`, 4 heads, feed-forward size 1024.
- **LSTM**: 3-layer LSTM with hidden size 512. The output goes through a linear projection back to the embedding size (256), which is needed for weight tying.

Most of the parameters in both models are in the embedding matrix (16384 × 256 ≈ 4.2M), the rest is the "body" of the network.

In [8]:
import time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.ln1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q, k, v = (t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2) for t in (q, k, v))
        att = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # causal mask
        x = x + self.drop(self.proj(att.transpose(1, 2).reshape(B, T, C)))
        x = x + self.drop(self.ff(self.ln2(x)))
        return x


class TransformerLM(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=256, n_layers=8, n_heads=4, d_ff=1024,
                 max_len=SEQ_LEN, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        nn.init.normal_(self.tok_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        self.head.weight = self.tok_emb.weight          # weight tying

    def forward(self, x, state=None):
        h = self.tok_emb(x) + self.pos_emb(torch.arange(x.size(1), device=x.device))
        h = self.drop(h)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_f(h)), None             # no state, same interface as the LSTM


class LSTMLM(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, emb_dim=256, hidden=512, n_layers=3, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb_dim, hidden, n_layers, batch_first=True, dropout=dropout)
        self.proj = nn.Linear(hidden, emb_dim)
        self.head = nn.Linear(emb_dim, vocab_size, bias=False)
        nn.init.normal_(self.emb.weight, std=0.02)
        self.head.weight = self.emb.weight               # weight tying

    def forward(self, x, state=None):
        h, state = self.lstm(self.drop(self.emb(x)), state)
        return self.head(self.proj(self.drop(h))), state


def count_params(model):
    return sum(p.numel() for p in model.parameters())   # shared weights are counted once

for name, cls in [("Transformer", TransformerLM), ("LSTM", LSTMLM)]:
    print(f"{name}: {count_params(cls()) / 1e6:.2f}M parameters")

Transformer: 10.58M parameters
LSTM: 10.11M parameters


### Batch size

The batch size is chosen by running one training step for several batch sizes and checking the peak GPU memory. Both models are trained with mixed precision (bfloat16). To keep the comparison fair, both models use the same batch size.

In [9]:
def peak_memory_gb(model_cls, batch_size):
    model = model_cls().to(device)
    opt = torch.optim.AdamW(model.parameters())
    x = torch.randint(0, VOCAB_SIZE, (batch_size, SEQ_LEN), device=device)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), x.reshape(-1))
        loss.backward()
        opt.step()
        return torch.cuda.max_memory_allocated() / 1e9
    except torch.cuda.OutOfMemoryError:
        return None
    finally:
        del model, opt
        torch.cuda.empty_cache()

print(f"{'batch':>6} {'Transformer':>12} {'LSTM':>8}   (peak GB)")
for bs in [32, 64, 128, 256]:
    mem_t, mem_l = peak_memory_gb(TransformerLM, bs), peak_memory_gb(LSTMLM, bs)
    fmt = lambda m: f"{m:.2f}" if m else "OOM"
    print(f"{bs:>6} {fmt(mem_t):>12} {fmt(mem_l):>8}")

 batch  Transformer     LSTM   (peak GB)
    32         2.33     1.85
    64         4.60     3.63
   128          OOM     7.21
   256          OOM      OOM


### Training

Both models are trained with the same setup: AdamW (lr 1e-3, weight decay 0.01), linear warmup followed by cosine decay, gradient clipping at 1.0, bfloat16 mixed precision, batch size 64. The loss is cross-entropy for the next token at every position. After each epoch we compute the loss on the validation set. Training time is measured only for the training loop.

In [10]:
import math

BATCH_SIZE = 64
EPOCHS = 3
LR = 1e-3

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)
print("steps per epoch:", len(train_dl))

@torch.no_grad()
def evaluate(model, dl):
    """Mean cross-entropy per token (in nats)."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(x)
        loss = F.cross_entropy(logits.float().reshape(-1, VOCAB_SIZE), y.reshape(-1), reduction="sum")
        total_loss += loss.item()
        total_tokens += y.numel()
    return total_loss / total_tokens


def train(model, epochs, name):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = epochs * len(train_dl)
    warmup = 200
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * s / total_steps)))

    history, train_time = [], 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        torch.cuda.synchronize()
        t0 = time.time()
        running = 0.0
        for x, y in tqdm(train_dl, desc=f"{name} epoch {epoch}"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits, _ = model(x)
            loss = F.cross_entropy(logits.float().reshape(-1, VOCAB_SIZE), y.reshape(-1))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            running += loss.item()
        torch.cuda.synchronize()
        train_time += time.time() - t0

        val_loss = evaluate(model, val_dl)
        history.append({"epoch": epoch, "train_loss": running / len(train_dl), "val_loss": val_loss})
        print(f"epoch {epoch}: train loss {running / len(train_dl):.3f} | val loss {val_loss:.3f} "
              f"| val ppl {math.exp(val_loss):.1f} | total train time {train_time / 60:.1f} min")

    torch.save(model.state_dict(), f"{name}.pt")
    return history, train_time

steps per epoch: 4328


In [11]:
torch.manual_seed(SEED)
transformer = TransformerLM()
hist_tr, time_tr = train(transformer, EPOCHS, "transformer")

transformer epoch 1: 100%|██████████████████████████████████████████████████████████████████████████████████| 4328/4328 [07:24<00:00,  9.73it/s]


epoch 1: train loss 4.800 | val loss 3.858 | val ppl 47.4 | total train time 7.4 min


transformer epoch 2: 100%|██████████████████████████████████████████████████████████████████████████████████| 4328/4328 [07:24<00:00,  9.73it/s]


epoch 2: train loss 3.803 | val loss 3.597 | val ppl 36.5 | total train time 14.8 min


transformer epoch 3: 100%|██████████████████████████████████████████████████████████████████████████████████| 4328/4328 [07:25<00:00,  9.72it/s]


epoch 3: train loss 3.634 | val loss 3.532 | val ppl 34.2 | total train time 22.2 min


In [12]:
torch.manual_seed(SEED)
lstm = LSTMLM()
hist_lstm, time_lstm = train(lstm, EPOCHS, "lstm")

lstm epoch 1: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 4328/4328 [07:06<00:00, 10.15it/s]


epoch 1: train loss 7.848 | val loss 6.436 | val ppl 623.8 | total train time 7.1 min


lstm epoch 2: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 4328/4328 [07:07<00:00, 10.14it/s]


epoch 2: train loss 5.336 | val loss 4.823 | val ppl 124.3 | total train time 14.2 min


lstm epoch 3: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 4328/4328 [07:06<00:00, 10.15it/s]


epoch 3: train loss 4.828 | val loss 4.698 | val ppl 109.8 | total train time 21.3 min


### Inference

The model outputs a probability distribution for the next token. We sample one token from it, append it to the text, and repeat. Top-k = 40 means we only take the 40 most probable tokens and sample from them, which cuts off rare, nonsensical tokens. A temperature of 0.8 sharpens the distribution, making the text less random.

Text is generated token by token with top-k sampling. The LSTM keeps its hidden state, so each step processes one token. The Transformer has no KV cache in this implementation, so each step recomputes the whole context (at most 256 tokens). This is the simplest implementation, but it makes generation slower for the Transformer, and it must be kept in mind when comparing the times.

Inference time is measured in two ways: (1) generation speed for a single sequence (tokens per second), (2) throughput of a full forward pass on a batch of 64 test sequences, which is how perplexity is computed.

In [29]:
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [30]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=60, temperature=0.8, top_k=40):
    model.eval()
    out = torch.tensor([sp.encode(prompt)], device=device)
    stateful = isinstance(model, LSTMLM)
    state = None
    for _ in range(max_new_tokens):
        if stateful:
            inp = out if state is None else out[:, -1:]     # LSTM: prompt once, then one token at a time
        else:
            inp = out[:, -SEQ_LEN:]                         # Transformer: whole context every time
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits, state = model(inp, state)
        logits = logits[0, -1].float() / temperature
        top_vals, top_idx = torch.topk(logits, top_k)
        probs = F.softmax(top_vals, dim=-1)
        next_id = top_idx[torch.multinomial(probs, 1)]
        out = torch.cat([out, next_id.view(1, 1)], dim=1)
    return sp.decode(out[0].tolist())


def gen_speed(model, prompt="Warszawa jest", n_tokens=200, repeats=3):
    generate(model, prompt, 10)                              # warm-up
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0 = time.time()
        generate(model, prompt, n_tokens)
        torch.cuda.synchronize()
        times.append(time.time() - t0)
    return n_tokens / (sum(times) / len(times))


@torch.no_grad()
def forward_speed(model, steps=20):
    model.eval()
    x = next(iter(test_dl))[0].to(device)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        model(x)                                             # warm-up
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(steps):
            model(x)
        torch.cuda.synchronize()
    return steps * x.numel() / (time.time() - t0)


eval_models = {"Transformer": transformer, "LSTM": lstm}
print(f"{'model':<16} {'generation [tok/s]':>20} {'forward pass [tok/s]':>22}")
for name, m in eval_models.items():
    m.to(device)
    print(f"{name:<16} {gen_speed(m):>20,.0f} {forward_speed(m):>22,.0f}")

model              generation [tok/s]   forward pass [tok/s]
Transformer                       506                980,680
LSTM                            1,062                806,901


### Perplexity

The test split (879 Wikipedia articles) is evaluated with the same `evaluate` function that was used for validation. Perplexity per token is $\exp(\text{loss})$, but it depends on the tokenizer, so we also report perplexity per word: the total loss of the text divided by the number of words, $\exp(\text{loss} \cdot N_{tokens} / N_{words})$.

In [31]:
final_models = {
    "Transformer": (transformer, hist_tr, time_tr),
    "LSTM": (lstm, hist_lstm, time_lstm),
}
tok_per_word = meta["test"]["tokens"] / meta["test"]["words"]

print(f"{'model':<12} {'params':>8} {'val ppl':>9} {'test loss':>10} {'test ppl/token':>15} {'test ppl/word':>14}")
for name, (m, hist, _) in final_models.items():
    m.to(device)
    loss = evaluate(m, test_dl)
    print(f"{name:<12} {count_params(m) / 1e6:>7.2f}M {math.exp(hist[-1]['val_loss']):>9.1f} "
          f"{loss:>10.3f} {math.exp(loss):>15.1f} {math.exp(loss * tok_per_word):>14.1f}")

model          params   val ppl  test loss  test ppl/token  test ppl/word
Transformer    10.58M      34.2      3.512            33.5          754.7
LSTM           10.11M     109.8      4.668           106.5         6673.0


The Transformer reaches a perplexity of 33.5 per token (about 755 per word), and the LSTM 106.5 (about 6700 per word). The test values are close to the validation ones (34.2 and 109.8), so the models are not overfitted to the validation set. Per word the numbers are much higher, because a Polish word is on average 1.89 tokens and the token losses add up.

The difference between the models is large, but it should not be treated as an exact measure of the architectures. Both models were trained with the same learning rate (1e-3) and for only 3 epochs, and the LSTM was still improving quickly at the end (validation perplexity 624, 124, 110 after the consecutive epochs), so it is probably undertrained. With a different learning rate or more epochs the gap would probably be smaller.

### Training time

Training time of the final models (only the training loop, validation excluded) together with the validation loss after every epoch.

In [32]:
tokens_per_epoch = len(train_dl) * BATCH_SIZE * SEQ_LEN

print(f"{'model':<12} {'params':>8} {'min/epoch':>10} {'total [min]':>12} {'tokens/s':>10}")
for name, (m, hist, t) in final_models.items():
    print(f"{name:<12} {count_params(m) / 1e6:>7.2f}M {t / EPOCHS / 60:>10.1f} {t / 60:>12.1f} "
          f"{EPOCHS * tokens_per_epoch / t:>10,.0f}")

print()
print("validation perplexity after each epoch:")
for name, (m, hist, t) in final_models.items():
    print(f"{name:<12}", [round(math.exp(h['val_loss']), 1) for h in hist])

model          params  min/epoch  total [min]   tokens/s
Transformer    10.58M        7.4         22.2    159,378
LSTM           10.11M        7.1         21.3    166,229

validation perplexity after each epoch:
Transformer  [47.4, 36.5, 34.2]
LSTM         [623.8, 124.3, 109.8]


One epoch of the Transformer takes 7.4 min and one epoch of the LSTM 5.9 min, so the Transformer is about 25% slower per epoch, although it has a similar number of parameters. However, time per epoch is not the whole story. After the first epoch (7.4 min) the Transformer already has a validation perplexity of 47, which the LSTM does not reach even after all three epochs (17.8 min, perplexity 75). So for a given training time the Transformer is clearly more efficient in terms of quality. The LSTM was still improving at the end of training, so with more epochs the gap would probably get smaller. All numbers come from a single run with one seed.

### Inference time

The generation speed and the forward pass throughput were measured in the cell above. Below we additionally check how the generation speed depends on the number of generated tokens, because in this implementation the Transformer has no KV cache and recomputes the whole context at every step, while the LSTM only processes the new token.

In [33]:
print(f"{'new tokens':>10} {'Transformer [tok/s]':>20} {'LSTM [tok/s]':>14}")
for n in [50, 100, 200]:
    speeds = [gen_speed(m, n_tokens=n) for m, _, _ in final_models.values()]
    print(f"{n:>10} {speeds[0]:>20,.0f} {speeds[1]:>14,.0f}")

new tokens  Transformer [tok/s]   LSTM [tok/s]
        50                  505          1,061
       100                  500          1,077
       200                  511          1,066


The LSTM generates about 1,060 tokens/s and the Transformer about 505 tokens/s, so the LSTM is about 2x faster. For both models the speed is practically constant for 50, 100 and 200 new tokens. For the Transformer this means that with a context of at most about 200 tokens the cost of attention is small, and a single step is dominated by the many small operations of its 8 layers (batch size 1, so the GPU is poorly utilized). The Transformer has no KV cache, so these numbers compare two simple implementations.

The two measurements point in opposite directions, so the result depends on the way the model is used: the LSTM is better for generating one sequence step by step, the Transformer for processing many sequences at once. Interestingly, the Transformer was slower in training (7.4 vs 5.9 min per epoch), although it is faster in the forward pass. 

### Prompts

Both models complete the same 10 Polish prompts (top-k sampling with k=40, temperature 0.8, 50 new tokens, the same random seed before each generation).

In [34]:
prompts = [
    "Kraków jest",
    "Adam Mickiewicz urodził się",
    "Druga wojna światowa rozpoczęła się",
    "Fotosynteza to proces",
    "Wisła jest najdłuższą rzeką",
    "AGH została założona w",
    "Mistrzostwa świata w piłce nożnej",
    "Język polski należy do",
    "Sieć neuronowa to",
    "Owczarek niemiecki",
]

for p in prompts:
    print("PROMPT:", p)
    for name, (m, _, _) in final_models.items():
        torch.manual_seed(SEED)
        print(f"  [{name}]", generate(m, p, max_new_tokens=50))
    print()

PROMPT: Kraków jest
  [Transformer] Kraków jest znany z wielu polskich i zagranicznych klubów – Czech, Czech i Czech. Najbardziej popularnym polskim klubem sportowym jest klub FK ŁKS Łódź. Historia. Klub założony został w roku 1898, pod nazwą NZR Rybnik. Od 1925 zespół występował w
  [LSTM] Kraków jest jego pierwszym właścicielem. Kubańskie Młyn – wieś w Polsce położona w województwie łódzkim, w powiecie bielsko-odłockim, w gminie Miki. W latach 1975–1998 miejscowość administracyjnie należała do województwa olsztyńskiego. Zabytki. Wieś

PROMPT: Adam Mickiewicz urodził się
  [Transformer] Adam Mickiewicz urodził się w rodzinie Antoniego i Anny z Borzycji. W 1852 uzyskał pracę dyplomową i został przyjęty do armii austriackiej i został przydzielony do I batalionu II batalionu II batalionu II batalionu III batalionu III batalionu III batalionu III batalionu III batalionu III batalionu III batalionu
  [LSTM] Adam Mickiewicz urodził się w Warszawie. W roku roku przeniósł się do Lublina, gdzi

Observations (about the outputs above):

- The Transformer writes text that looks like a Wikipedia article: correct inflection and word order, typical sentence structures ("X – miasto w ...", "w roku ... "), and the continuation usually fits the prompt. The facts are often wrong, though (dates, places, names), which is expected from such a small model trained for a short time. The model knows the style much better than the content.
- The LSTM also produces Polish words, but the text is less coherent: more errors in inflection, sentences that drift away from the topic and repetitions of phrases. It keeps the local structure, but loses the thread after a few words.
- This is in line with the perplexity: the model with perplexity 34 per token produces visibly better text than the one with 75. On the other hand, none of the models can be used as a source of knowledge, the completions only show how well the language and the style were learned.